# Hypothesis Space II: From Fixed Features to Deep Representations

## 0. Learning objectives and lesson structure

We connect the previous hypothesis-space notebook to deep learning. The central question is:

$$
\text{what changes when the feature map } \phi \text{ itself becomes learned?}
$$

Previously, a learner selected a function from a fixed hypothesis space

$$
\mathcal{H}_{\phi}
= \left\{x \mapsto \theta^{\top}\phi(x) : \theta \in \Theta\right\}.
$$

In deep learning, the representation is also parameterised:

$$
\mathcal{H}_{\text{NN}}
= \left\{x \mapsto h_{\theta}(x) : \theta = (W_1,b_1,\ldots,W_L,b_L)\right\}.
$$

By the end, students should be able to:
- define a neural-network hypothesis space as a parameterised family of composed functions;
- explain the role of nonlinear activations;
- interpret width as adding learned features;
- interpret depth as composition and reuse;
- distinguish expressivity, approximation, optimisation, and generalisation;
- explain how architecture imposes inductive bias;
- describe how data support and hypothesis-space bias shape interpolation and extrapolation.

In [ ]:
# Setup cell for the notebook.
#
# This cell should eventually:
# - import numpy, matplotlib, and IPython display utilities;
# - optionally import ipywidgets if the notebook will use sliders;
# - create a shared one-dimensional input grid, for example x_grid in [-2, 2];
# - define a small plotting helper that clears figures cleanly after display;
# - set a random seed so every classroom run uses the same examples.
#
# Interaction notes:
# - Run this once at the start of the notebook.
# - Later cells should reuse the same x_grid and plotting helper.
# - Keep examples small enough that sliders update immediately in Colab or Jupyter.

## 1. From fixed feature maps to learned representations

In a fixed-feature model, the design choice is the feature map and learning only selects coefficients:

$$
h_{\theta}(x) = \theta^{\top}\phi(x),
\qquad
\theta \in \mathbb{R}^{p}.
$$

The corresponding hypothesis space is

$$
\mathcal{H}_{\phi}
= \left\{h_{\theta}: h_{\theta}(x)=\theta^{\top}\phi(x),\ \theta\in\mathbb{R}^{p}\right\}.
$$

For example, a polynomial feature map might be

$$
\phi_d(x) = \begin{bmatrix}1 & x & x^2 & \cdots & x^d\end{bmatrix}^{\top},
$$

so changing \(\theta\) changes the selected polynomial, while changing \(d\) changes the hypothesis space itself.

A neural network learns intermediate representations. With \(z_0=x\), a feed-forward network can be written as

$$
z_{\ell} = \sigma(W_{\ell}z_{\ell-1}+b_{\ell}),
\qquad \ell=1,\ldots,L-1,
$$

and

$$
h_{\theta}(x)=W_L z_{L-1}+b_L.
$$

Equivalently,

$$
h_{\theta}
= g_L \circ g_{L-1} \circ \cdots \circ g_1,
$$

where each \(g_{\ell}\) is a learned transformation. This enlarges \(\mathcal{H}\) by making feature construction adaptive.

In [ ]:
# Example cell: fixed polynomial features versus learned hidden features.
#
# This cell should eventually:
# - define a fixed polynomial basis phi_d(x) = [1, x, ..., x^d];
# - plot several functions theta^T phi_d(x) obtained by changing theta while d is fixed;
# - contrast this with a small neural feature map where w and b move the hidden features;
# - make clear which controls change the selected function and which controls change H.
#
# Suggested interaction:
# - Use a degree slider d for the polynomial feature map.
# - Use coefficient sliders theta_0, theta_1, ..., theta_d for the selected polynomial.
# - For the learned-feature view, use sliders for hidden-feature parameters w_j and b_j.
# - Ask students: did this action move within H, or did it redefine H?

## 2. One neuron as a learned nonlinear feature

A single neuron defines a learned feature

$$
a_{w,b}(x)=\sigma(w^{\top}x+b),
$$

where \(w\) controls orientation and scale, \(b\) shifts the activation boundary, and \(\sigma\) is the activation function.

For ReLU,

$$
\sigma(t)=\max(0,t),
$$

so in one dimension

$$
a_{w,b}(x)=\max(0,wx+b).
$$

If \(w\neq 0\), the kink occurs at

$$
x^{\star}=-\frac{b}{w}.
$$

In more than one dimension, the activation boundary is the hyperplane

$$
\left\{x : w^{\top}x+b=0\right\}.
$$

The nonlinearity is essential. Without \(\sigma\), two linear layers collapse into one linear map:

$$
W_2(W_1x+b_1)+b_2 = (W_2W_1)x + (W_2b_1+b_2).
$$

Stacking linear layers therefore does not create a richer hypothesis space.

In [ ]:
# Example cell: one ReLU neuron as a learned feature.
#
# This cell should eventually:
# - define relu(t) = max(0, t);
# - compute the preactivation t(x) = w*x + b on the shared x_grid;
# - compute the activation a(x) = relu(t(x));
# - plot both t(x) and a(x), marking the kink x_star = -b / w when w is nonzero;
# - show that changing b shifts the boundary and changing w changes slope/orientation.
#
# Suggested interaction:
# - Use sliders for w and b.
# - Ask students to predict where the kink will move before changing b.
# - Include a checkbox that toggles the activation on/off so students can see the
#   difference between a linear feature and a nonlinear learned feature.

## 3. Width: combining many learned features

A one-hidden-layer network with \(m\) hidden units can be written as

$$
h(x)=c+\sum_{j=1}^{m} v_j\,\sigma(w_jx+b_j).
$$

For ReLU in one dimension,

$$
h(x)=c+\sum_{j=1}^{m} v_j\max(0,w_jx+b_j),
$$

which is a learned piecewise-linear function. Each active hidden unit contributes a hinge at

$$
x_j^{\star}=-\frac{b_j}{w_j},
$$

when \(w_j\neq 0\). The output weights \(v_j\) decide how strongly each learned feature contributes to the final prediction.

The width-limited hypothesis space is

$$
\mathcal{H}_{m}
= \left\{x \mapsto c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j) : c,v_j,w_j,b_j\in\mathbb{R}\right\}.
$$

Increasing \(m\) adds more learned features. Universal approximation results say that sufficiently wide networks can approximate broad classes of continuous functions on compact domains, for example

$$
\forall f\in C(K),\ \forall \varepsilon>0,\ \exists h\in\mathcal{H}_{m}
\text{ such that }
\sup_{x\in K}|f(x)-h(x)|<\varepsilon,
$$

for suitable activations and large enough \(m\). This is an approximation statement, not a guarantee that training will find the function or that it will generalise.

In [ ]:
# Example cell: width as a sum of learned ReLU features.
#
# This cell should eventually:
# - create three ReLU neurons a_j(x) = relu(w_j*x + b_j);
# - plot each weighted contribution v_j * a_j(x) as a faint line;
# - plot the summed output h(x) = c + sum_j v_j * a_j(x) as the main line;
# - mark each kink x_j_star = -b_j / w_j when visible in the plotting domain;
# - optionally compare m = 1, 3, and 8 to show how width adds hinges.
#
# Suggested interaction:
# - Use sliders for w_j, b_j, v_j, and c.
# - Let students manipulate one neuron at a time and identify which segment of h(x)
#   changes as a result.
# - Include a reset button or preset parameter sets for common shapes such as V-shapes,
#   ramps, and shallow piecewise-linear approximations to smooth curves.

## 4. Depth: composition and reuse

A deep feed-forward network repeatedly transforms a representation:

$$
z_0=x,
\qquad
z_{\ell}=\sigma(W_{\ell}z_{\ell-1}+b_{\ell}),
\qquad \ell=1,\ldots,L-1,
$$

with output

$$
h_{\theta}(x)=W_Lz_{L-1}+b_L.
$$

Depth matters because later layers reuse and recombine earlier features. Written as a composition,

$$
h_{\theta}(x)=g_L(g_{L-1}(\cdots g_1(x)\cdots)).
$$

For ReLU networks, each activation pattern defines a region \(R_r\) of input space. Inside one such region, the network is affine:

$$
h_{\theta}(x)=A_rx+c_r,
\qquad x\in R_r.
$$

Depth can create many such regions efficiently. This means some compositional functions can be represented with far fewer units by a deep network than by a shallow one.

In [ ]:
# Example cell: depth as composition and feature reuse.
#
# This cell should eventually:
# - define a small two-layer or three-layer ReLU network in one dimension;
# - compute and plot intermediate representations z_1(x), z_2(x), and h(x);
# - show how later layers reuse earlier kinks/features rather than starting from x alone;
# - compare a shallow network and a deeper network with a similar number of parameters;
# - optionally count visible linear regions over the plotting grid.
#
# Suggested interaction:
# - Use preset buttons for shallow, deep, and linear-only configurations.
# - Let students toggle individual layers on/off and observe how the final function changes.
# - Ask students to identify whether a new bend came from a new first-layer feature or
#   from recombining features in a later layer.

## 5. Architecture as inductive bias

An architecture is not only an implementation choice. It changes the functions that are available, efficient, and natural inside \(\mathcal{H}\).

Common examples:

- MLP: flexible maps such as \(h(x)=W_L\sigma(\cdots\sigma(W_1x+b_1)\cdots)+b_L\), with weak structural assumptions.
- CNN: shared local filters, often written in one dimension as

$$
z_{i,k}=\sigma\left(\sum_{\Delta,c}K_{\Delta,c,k}x_{i+\Delta,c}+b_k\right),
$$

which encourages locality and translation equivariance, approximately

$$
F(T_{\tau}x)=T_{\tau}F(x).
$$

- RNN: sequential recurrence

$$
h_t=f_{\theta}(x_t,h_{t-1}),
$$

which reuses the same transition across time.

- Transformer: learned token-to-token interactions through attention

$$
\operatorname{Attention}(Q,K,V)
=\operatorname{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V.
$$

- GNN: relational message passing, for example

$$
h_v^{(\ell+1)}=\psi_{\theta}\left(h_v^{(\ell)},\sum_{u\in\mathcal{N}(v)}\phi_{\theta}(h_u^{(\ell)},h_v^{(\ell)},e_{uv})\right).
$$

A good architecture makes useful functions easier to learn and irrelevant functions harder to learn.

In [ ]:
# Example cell: architecture as inductive bias.
#
# This cell should eventually use lightweight demonstrations rather than training
# full neural networks.
#
# Possible demonstrations:
# - MLP: show dense mixing where every input coordinate can affect every hidden unit.
# - CNN: create a one-dimensional signal, apply the same local filter at every position,
#   then shift the input and show the output shifts with it.
# - RNN: process a short sequence with the same update rule at every time step.
# - Transformer: display a small attention matrix over tokens and show how changing one
#   query changes the weighted combination of values.
# - GNN: pass messages over a tiny graph and show that node updates depend on neighbours.
#
# Suggested interaction:
# - Use toggles for architecture type.
# - Keep the same toy input and ask which transformations each architecture makes natural.
# - Avoid large training runs; the goal is to expose bias, not benchmark models.

## 6. Data support, interpolation, and extrapolation

Let the observed dataset be

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n}.
$$

Finite data constrain a hypothesis only where observations provide evidence. The empirical risk is

$$
\widehat{R}_{\mathcal{D}}(h)
=\frac{1}{n}\sum_{i=1}^{n}\ell(h(x_i),y_i),
$$

so two functions can have similar empirical risk while disagreeing away from the observed inputs. The compatible set

$$
\left\{h\in\mathcal{H}: \widehat{R}_{\mathcal{D}}(h)\leq \epsilon\right\}
$$

may contain many very different functions.

Within well-supported regions, many hypotheses may be plausible but constrained. Away from support, behaviour is dominated by hypothesis-space, regularisation, and optimisation biases.

Examples:

- Polynomial features:

$$
p_d(x)=\sum_{k=0}^{d}\theta_kx^k
$$

can extrapolate unstably when \(d\) is large or the coordinates are poorly conditioned.

- A one-dimensional ReLU network extrapolates affinely once no new activation boundaries are crossed:

$$
h(x)=A_rx+c_r \quad \text{inside region } R_r.
$$

- Periodic features such as

$$
h(x)=a_0+\sum_{k=1}^{K}\left(a_k\sin(2\pi kx)+b_k\cos(2\pi kx)\right)
$$

extrapolate periodically by construction.

In [ ]:
# Example cell: support, interpolation, and extrapolation.
#
# This cell should eventually:
# - generate a fixed regression dataset with a deliberate gap or partial domain coverage;
# - shade observed/support regions and unsupported extrapolation regions on the plot;
# - fit or display several hypothesis classes: low-degree polynomial, high-degree
#   polynomial, ReLU piecewise-linear model, and periodic basis model;
# - compare how the functions agree near data but diverge in gaps or outside support;
# - report empirical error on observed points and, if using simulated data, oracle error
#   on a dense grid for teaching purposes.
#
# Suggested interaction:
# - Use controls for polynomial degree, ReLU width, periodic frequency count, and noise.
# - Keep the dataset fixed while changing H so students can isolate hypothesis-space effects.
# - Ask students which extrapolation they would trust and what assumption justifies that trust.

## 7. What expressivity does not solve

Expressivity asks whether the target function can be represented inside the hypothesis space:

$$
\inf_{h\in\mathcal{H}}R(h),
\qquad
R(h)=\mathbb{E}_{(X,Y)\sim P}\left[\ell(h(X),Y)\right].
$$

But this is only one issue. The learning problem also involves:

- representation: is there an \(h\in\mathcal{H}\) with small risk?
- optimisation: can training find a good parameter vector?

$$
\widehat{\theta}\approx\arg\min_{\theta\in\Theta}\widehat{R}_{\mathcal{D}}(h_{\theta}).
$$

- identification: can finite data distinguish between plausible functions?

$$
h_1(x_i)=h_2(x_i)\ \forall i
\quad\not\Rightarrow\quad
h_1(x)=h_2(x)\ \forall x.
$$

- generalisation: will empirical performance transfer to new data?

$$
R(\widehat{h})-\widehat{R}_{\mathcal{D}}(\widehat{h}).
$$

Large networks can fit arbitrary labels when they have enough capacity, so expressivity alone does not explain good generalisation.

In [ ]:
# Example cell: expressivity is not the whole learning problem.
#
# This cell should eventually:
# - create a train/test split from a simple regression dataset;
# - fit or simulate models with increasing capacity;
# - plot training error, test error, and optionally oracle/grid error against capacity;
# - include a random-label toggle to show that high-capacity models can interpolate
#   labels without learning a useful relationship;
# - separate approximation failure, optimisation failure, identification ambiguity,
#   and generalisation failure in the printed summary.
#
# Suggested interaction:
# - Ask students to predict what happens to training error as capacity increases.
# - Then ask whether lower training error necessarily means better deployment behaviour.
# - Use the random-label condition as a discussion prompt, not as a full benchmark.

## 8. Parameter space versus hypothesis space

The parameter vector \(\theta\) is a coordinate in parameter space \(\Theta\). The realised function \(h_{\theta}\) is an element of hypothesis space \(\mathcal{H}\). The map

$$
q:\Theta\to\mathcal{H},
\qquad
q(\theta)=h_{\theta},
$$

need not be one-to-one. Different parameter values can represent the same function.

For a one-hidden-layer network,

$$
h(x)=c+\sum_{j=1}^{m}v_j\sigma(w_jx+b_j),
$$

permuting hidden units leaves the function unchanged. For any permutation \(\pi\),

$$
\sum_{j=1}^{m}v_j\sigma(w_jx+b_j)
=
\sum_{j=1}^{m}v_{\pi(j)}\sigma(w_{\pi(j)}x+b_{\pi(j)}).
$$

ReLU also has positive homogeneity. For \(\alpha>0\),

$$
v_j\sigma(w_jx+b_j)
=
\frac{v_j}{\alpha}\sigma(\alpha w_jx+\alpha b_j).
$$

Optimisation moves through \(\Theta\), but prediction happens in function space.

In [ ]:
# Example cell: many parameter vectors can represent the same function.
#
# This cell should eventually:
# - define a small one-hidden-layer ReLU network with two or three hidden units;
# - compute h_theta(x) for an original parameter set;
# - create a second parameter set by permuting hidden units and show the plotted
#   function is identical;
# - create a third parameter set using ReLU positive rescaling and show the function
#   is still identical up to numerical precision;
# - print max_abs_difference between the original and transformed functions.
#
# Suggested interaction:
# - Let students change parameters directly and distinguish parameter movement from
#   function movement.
# - Ask why optimisation paths in parameter space can look different even when the
#   final predictive function is the same.

## 9. Summary and link to optimisation space

The workshop frame remains

$$
\boxed{\mathcal{H}} + \mathcal{D} + \mathcal{O} \rightarrow s.
$$

In this notebook:

$$
\mathcal{H}\quad\text{defines what is possible,}
$$

$$
\mathcal{D}\quad\text{defines what evidence is available,}
$$

and

$$
\mathcal{O}\quad\text{determines which hypothesis is selected.}
$$

A typical selection rule has the form

$$
\widehat{h}
=\arg\min_{h\in\mathcal{H}}
\left[\widehat{R}_{\mathcal{D}}(h)+\lambda\Omega(h)\right],
$$

where \(\widehat{R}_{\mathcal{D}}\) measures fit to observed data and \(\Omega\) expresses a penalty or preference.

The next notebook asks:

$$
\text{Given a large }\mathcal{H},\text{ how does training choose one function from many?}
$$

In [ ]:
# Optional wrap-up cell.
#
# This cell should eventually:
# - display a compact table connecting each notebook ingredient to a classroom example:
#   H: feature maps, width, depth, architecture;
#   D: observed support, gaps, noise;
#   O: loss, regularisation, optimiser, stopping rule;
# - optionally ask students to choose one plotted function and identify the assumptions
#   that make it plausible.
#
# Suggested interaction:
# - Use this as an exit ticket or short discussion prompt.
# - Do not add new machinery here; summarise the evidence from previous cells.